<a href="https://colab.research.google.com/github/vladislavpykhteev/HSE_Vladislav_Pykhteev/blob/main/lesson_11/lesson_11.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Домашнее задание: Юридический скрининг реестра контрагентов

Юротдел компании готовится к партнёрскому собранию. Перед ним — реестр контрагентов, с которыми компания планирует работу в следующем квартале: новые поставщики, заёмщики, инвестиционные партнёры.

До собрания нужно провести **юридический скрининг реестра**:

1. Каждого контрагента проверить по **чёрному списку Банка России** (компании с признаками нелегальной деятельности на финансовом рынке).
2. Для контрагентов с валютными договорами — определить **курс ЦБ на дату платежа** и зафиксировать расхождение «запрошенная дата / фактическая дата курса».
3. Присвоить каждому контрагенту **уровень риска** по правилам, которые вы сформулируете сами.
4. Подготовить **материалы для собрания**: Excel-дашборд и текстовое резюме с числами.

Реестр задан в ячейке ниже. Работайте с ним как с исходными данными: менять ИНН, суммы и даты нельзя, всё остальное — в ваших руках.

## Исходные данные

**Список контрагентов**. Поля: `name` (название), `inn` (ИНН, строка), `region` (субъект РФ), `contract_type` (тип договора), `amount` (сумма обязательства, руб.), `payment_date` (дата платежа по договору, `ДД.ММ.ГГГГ`).

⚠️ Данные реестра намеренно неоднородны: ИНН — строки, среди них встречаются записи с пробелами и лишними символами; суммы — целые числа; даты платежей — строки российского формата, часть дат попадает на выходные и праздники.

In [1]:
counterparties = [
    {"name": "ООО «ДРАЙВ ТРАСТ»",                 "inn": "7730311629",  "region": "Москва",                   "contract_type": "заём",       "amount": 4_500_000,  "payment_date": "10.03.2025"},
    {"name": "ООО «ЦПН КВ Групп»",                "inn": "7806622850",  "region": "Санкт-Петербург",          "contract_type": "инвестиции", "amount": 12_000_000, "payment_date": "15.04.2025"},
    {"name": "КПК «Центр финансового содействия»","inn": "1435279293",  "region": "Республика Саха (Якутия)", "contract_type": "заём",       "amount": 2_800_000,  "payment_date": "20.05.2025"},
    {"name": "ПК «ПОТОК»",                        "inn": "9204006374",  "region": "Республика Крым",          "contract_type": "инвестиции", "amount": 7_300_000,  "payment_date": "01.07.2025"},
    {"name": "ООО «УК Селигер 24»",               "inn": "6913017372",  "region": "Тверская область",         "contract_type": "инвестиции", "amount": 5_600_000,  "payment_date": "17.03.2025"},
    {"name": "ПК «Платформа Ру»",                 "inn": "7733473794",  "region": "Москва",                   "contract_type": "инвестиции", "amount": 9_900_000,  "payment_date": "23.02.2025"},
    {"name": "КПК «Сберкнижка»",                  "inn": "6452126313",  "region": "Саратовская область",      "contract_type": "заём",       "amount": 1_750_000,  "payment_date": "09.05.2025"},
    {"name": "СК «Образец»",                      "inn": "0571011790",  "region": "Чеченская Республика",     "contract_type": "страхование","amount": 3_250_000,  "payment_date": "16.06.2025"},
    {"name": "ООО «СтройМонтажСервис»",           "inn": " 7707083893", "region": "Москва",                   "contract_type": "подряд",     "amount": 9_500_000,  "payment_date": "12.04.2025"},
    {"name": "АО «ТранспортЛизинг»",              "inn": "5029066329",  "region": "Московская область",       "contract_type": "лизинг",     "amount": 6_200_000,  "payment_date": "28.04.2025"},
    {"name": "ООО «АгроТрейд Юг»",                "inn": "2310035747",  "region": "Краснодарский край",       "contract_type": "поставка",   "amount": 3_400_000,  "payment_date": "05.05.2025"},
    {"name": "ООО «ИТ-Интегратор»",               "inn": "7810142190",  "region": "Санкт-Петербург",          "contract_type": "услуги",     "amount": 1_900_000,  "payment_date": "19.05.2025"},
    {"name": "ООО «УралМетСнаб»",                 "inn": "6674287672",  "region": "Свердловская область",     "contract_type": "поставка",   "amount": 15_800_000, "payment_date": "26.04.2025"},
    {"name": "ООО «ВолгаСтрой»",                  "inn": "6315027260",  "region": "Самарская область",        "contract_type": "подряд",     "amount": 8_100_000,  "payment_date": "01.06.2025"},
    {"name": "ООО «НижегородОплет»",              "inn": "5260474345",  "region": "Нижегородская область",    "contract_type": "поставка",   "amount": 950_000,    "payment_date": "10.05.2025"},
    {"name": "ООО «СибЭнергоКомплект»",           "inn": "7709803000",  "region": "Москва",                   "contract_type": "услуги",     "amount": 4_100_000,  "payment_date": "22.06.2025"},
    {"name": "АО «БалтФлот Сервис»",              "inn": "7811018401",  "region": "Санкт-Петербург",          "contract_type": "услуги",     "amount": 2_450_000,  "payment_date": "14.01.2025"},
    {"name": "ООО «Средне-Волжский Завод»",       "inn": "6321049153",  "region": "Самарская область",        "contract_type": "поставка",   "amount": 11_300_000, "payment_date": "07.02.2025"},
    {"name": "ООО «СибирьАгроМаш»",               "inn": "5504001577",  "region": "Омская область",           "contract_type": "поставка",   "amount": 5_750_000,  "payment_date": "01.03.2025"},
    {"name": "ООО «КубаньАгроХим»",               "inn": "2366375063",  "region": "Краснодарский край",       "contract_type": "поставка",   "amount": 7_050_000,  "payment_date": "04.07.2025"},
    {"name": "ООО «КазаньСтройПроект»",           "inn": "1651007245",  "region": "Республика Татарстан",     "contract_type": "подряд",     "amount": 13_200_000, "payment_date": "15.08.2025"},
    {"name": "ООО «ВоронежТеплоСеть»",            "inn": "3664069397",  "region": "Воронежская область",      "contract_type": "услуги",     "amount": 2_150_000,  "payment_date": "07.10.2025"},
    {"name": "ООО «АлтайЭнергоПром»",             "inn": "2225006811",  "region": "Алтайский край",           "contract_type": "подряд",     "amount": 6_850_000,  "payment_date": "04.11.2025"},
    {"name": "ООО «Северный Порт»",               "inn": "2902098681",  "region": "Архангельская область",    "contract_type": "поставка",   "amount": 3_980_000,  "payment_date": "15.11.2025"},
    {"name": "ООО «ДальРыбФлот»",                 "inn": "2540166843",  "region": "Приморский край",          "contract_type": "поставка",   "amount": 18_400_000, "payment_date": "31.12.2025"},
    {"name": "ООО «РостовАгроСнаб»",              "inn": "6165094039",  "region": "Ростовская область",       "contract_type": "поставка",   "amount": 4_600_000,  "payment_date": "30.10.2025"},
    {"name": "ООО «ЕкатеринбургСигнал»",          "inn": "6671180619",  "region": "Свердловская область",     "contract_type": "услуги",     "amount": 1_380_000,  "payment_date": "21.11.2025"},
    {"name": "ООО «Жуковский АвиаТех»",           "inn": "5013025761",  "region": "Московская область",       "contract_type": "подряд",     "amount": 10_600_000, "payment_date": "12.06.2025"},
    {"name": "ООО «ЧелябинскМеталлТорг»",         "inn": "7444800868",  "region": "Челябинская область",      "contract_type": "поставка",   "amount": 8_900_000,  "payment_date": "01.09.2025"},
    {"name": "ООО «ЕкатеринбургСтройСнаб»",       "inn": "6673009275",  "region": "Свердловская область",     "contract_type": "подряд",     "amount": 5_300_000,  "payment_date": "30.10.2025"},
]

print(f"Контрагентов в реестре: {len(counterparties)}")

Контрагентов в реестре: 30


---
## Задание

Выполняйте блоки по порядку: выход каждого блока — вход следующего. Код пишите в ячейках с пометкой **«ВАШ КОД»**.

Требования ко всему коду:

- Все запросы к API — с `timeout` и `raise_for_status()`.
- Любой сетевой сбой не должен останавливать обработку реестра: ошибка фиксируется, обработка продолжается.
- Между запросами к API — пауза (`time.sleep`).
- Имена переменных и функций — на английском, комментарии — на русском.
- Код выполняется сверху вниз без ошибок.

---
## Блок 1. Скрининг по чёрному списку ЦБ

Проверьте **каждого** контрагента реестра по чёрному списку Банка России.

Что нужно сделать:

1. **Подготовить ИНН к проверке.** В реестре ИНН заданы строками, и не все они чистые: где-то лишний пробел, где-то другой «мусор». Приведите все ИНН к единому виду — строка из цифр без пробелов и посторонних символов. Добавьте в каждую запись поле `inn_clean`. Решите сами, как проверить корректность ИНН перед запросом. Некорректный ИНН — это не повод останавливать скрининг.
2. **Проверить каждый ИНН через API чёрного списка** (`Search` по фразе → при наличии кандидатов — `DetailInfo` → **точная сверка ИНН**). Помните: поиск по фразе находит подстроку — совпадение по фразе ещё не означает, что это та же организация.
3. **Зафиксировать результат проверки** для каждого контрагента: найден/не найден, а для найденных — название из реестра ЦБ, признаки, дата внесения, ликвидирована ли организация.
4. **Обеспечить устойчивость**: если запрос не удался (сеть, таймаут, ошибка сервера) — зафиксируйте сбой и продолжите проверку остальных. В конце выведите статистику: сколько проверено, сколько найдено в списке, сколько сбоев.

После блока у каждой записи реестра должны появиться поля с результатом проверки. Выведите итоговую таблицу: контрагент — статус проверки.

In [5]:
# ===================== БЛОК 1 — ВАШ КОД =====================
import re
import time
from collections import Counter

import requests
import pandas as pd
from requests.exceptions import RequestException

WARNING_API = "https://www.cbr.ru/warninglistapi"
REQUEST_TIMEOUT = 15
REQUEST_DELAY = 0.25

# 1. Очистка ИНН → поле inn_clean
def clean_inn(raw_inn):
    return re.sub(r"\D", "", str(raw_inn))
def is_valid_inn(inn):
    return inn.isdigit() and len(inn) in (10, 12)
def has_valid_inn_checksum(inn):
    if not is_valid_inn(inn):
        return False
    # Проводим проверку контрольного числа, предварительно преобразовав строку в число
    digits = [int(char) for char in inn]
    def checksum(coefficients):
        return sum(value * coefficient for value, coefficient in zip(digits, coefficients)) % 11 % 10
    if len(inn) == 10:
        return checksum((2, 4, 10, 3, 5, 9, 4, 6, 8)) == digits[9]
    first_check = checksum((7, 2, 4, 10, 3, 5, 9, 4, 6, 8)) == digits[10]
    second_check = checksum((3, 7, 2, 4, 10, 3, 5, 9, 4, 6, 8)) == digits[11]
    return first_check and second_check

# Нормализация ответов API о ликвидации
def normalize_bool(value):
    if isinstance(value, bool):
        return value
    return str(value).strip().lower() in {"true", "1", "да", "yes"}

# 2. Проверка каждого ИНН через API (Search → DetailInfo → точная сверка)
# Ищет ИНН через Search, затем подтверждает его через DetailInfo
# Возвращает словарь с единым набором полей даже при сетевой ошибке
def check_counterparty(inn):
    result = {
        "status": "не найден",
        "found": False,
        "registry_name": None,
        "signs": [],
        "listed_date": None,
        "liquidated": None,
        "error": None,
    }
    try:
        # Сначала Search возвращает только краткий список возможных совпадений
        search_response = requests.get(
            f"{WARNING_API}/Search",
            params={"sphrase": inn, "page": 0},
            timeout=REQUEST_TIMEOUT,
        )
        search_response.raise_for_status()
        candidates = search_response.json().get("Data", [])
        time.sleep(REQUEST_DELAY)  # пауза после запроса Search

        for candidate in candidates:
            # Затем по id открываем полную карточку каждого контрагента
            detail_response = requests.get(
                f"{WARNING_API}/DetailInfo",
                params={"id": candidate["id"]},
                timeout=REQUEST_TIMEOUT,
            )
            detail_response.raise_for_status()
            detail = detail_response.json()
            time.sleep(REQUEST_DELAY)  # пауза после каждого DetailInfo

            if not detail.get("Info"):
                continue
            info = detail["Info"][0]
            detail_inn = clean_inn(info.get("inn", ""))
            # Search ищет подстроку, поэтому найденным считаем только точное совпадение
            if detail_inn == inn:
                signs = [sign.get("signRus", "") for sign in detail.get("Signs", [])]
                result.update({
                    "status": "найден в чёрном списке",
                    "found": True,
                    "registry_name": info.get("nameOrg"),
                    "signs": [sign for sign in signs if sign],
                    "listed_date": str(info.get("dt", ""))[:10] or None,
                    "liquidated": normalize_bool(info.get("isLikvid")),
                })
                return result

        return result
    # Ошибку сохраняем в данных, поэтому один сбой не остановит весь реестр
    except (RequestException, ValueError, KeyError, TypeError) as error:
        result.update({
            "status": "ошибка API",
            "found": None,
            "error": f"{type(error).__name__}: {error}",
        })
        return result
# В исходном списке есть пробелы и другие символы, поэтому сохраняем очищенный вариант отдельно
for counterparty in counterparties:
    counterparty["inn_clean"] = clean_inn(counterparty["inn"])
    counterparty["inn_valid"] = is_valid_inn(counterparty["inn_clean"])
    counterparty["inn_checksum_valid"] = has_valid_inn_checksum(counterparty["inn_clean"])
    # Фиксируем запись с неверным ИНН
    if not counterparty["inn_valid"]:
        check_result = {
            "status": "некорректный ИНН",
            "found": None,
            "registry_name": None,
            "signs": [],
            "listed_date": None,
            "liquidated": None,
            "error": "ИНН не состоит из 10 или 12 цифр",
        }
    else:
        check_result = check_counterparty(counterparty["inn_clean"])

    # 3. Фиксация результата проверки в записи реестра
    counterparty["cb_status"] = check_result["status"]
    counterparty["cb_found"] = check_result["found"]
    counterparty["cb_name"] = check_result["registry_name"]
    counterparty["cb_signs"] = check_result["signs"]
    counterparty["cb_listed_date"] = check_result["listed_date"]
    counterparty["cb_liquidated"] = check_result["liquidated"]
    counterparty["cb_error"] = check_result["error"]

# 4. Устойчивость к сбоям + итоговая статистика
processed_count = len(counterparties)
successful_checks = sum(cp["cb_status"] in {"не найден", "найден в чёрном списке"} for cp in counterparties)
found_count = sum(cp["cb_found"] is True for cp in counterparties)
failure_count = sum(cp["cb_status"] in {"ошибка API", "некорректный ИНН"} for cp in counterparties)
print(f"Обработано записей: {processed_count}")
print(f"Успешно проверено через API: {successful_checks}")
print(f"Найдено в чёрном списке: {found_count}")
print(f"Сбоев либо некорректных ИНН: {failure_count}")

screening_table = pd.DataFrame([
    {
        "Контрагент": cp["name"],
        "ИНН": cp["inn_clean"],
        "ИНН корректен": cp["inn_valid"],
        "Контрольная сумма ИНН": cp["inn_checksum_valid"],
        "Статус проверки": cp["cb_status"],
        "Название в реестре ЦБ": cp["cb_name"],
        "Признаки": "; ".join(cp["cb_signs"]),
        "Дата внесения": cp["cb_listed_date"],
        "Ликвидировано": cp["cb_liquidated"],
        "Ошибка": cp["cb_error"],
    }
    for cp in counterparties
])
display(screening_table)


Обработано записей: 30
Успешно проверено через API: 30
Найдено в чёрном списке: 8
Сбоев либо некорректных ИНН: 0


,Контрагент,ИНН,ИНН корректен,Контрольная сумма ИНН,Статус проверки,Название в реестре ЦБ,Признаки,Дата внесения,Ликвидировано,Ошибка
0,ООО «ДРАЙВ ТРАСТ»,7730311629,True,True,найден в чёрном списке,"ОБЩЕСТВО С ОГРАНИЧЕННОЙ ОТВЕТСТВЕННОСТЬЮ ""ДРАЙ...",Признаки нелегального кредитора,2026-09-11,False,None
1,ООО «ЦПН КВ Групп»,7806622850,True,True,найден в чёрном списке,"ОБЩЕСТВО С ОГРАНИЧЕННОЙ ОТВЕТСТВЕННОСТЬЮ ""ЦПН ...",Признаки «финансовой пирамиды»,2026-09-08,False,None
2,КПК «Центр финансового содействия»,1435279293,True,True,найден в чёрном списке,"КРЕДИТНЫЙ ПОТРЕБИТЕЛЬСКИЙ КООПЕРАТИВ ""ЦЕНТР ФИ...",Признаки «финансовой пирамиды»; Признаки нелег...,2026-09-04,False,None
3,ПК «ПОТОК»,9204006374,True,True,найден в чёрном списке,"ПОТРЕБИТЕЛЬСКИЙ КООПЕРАТИВ ""ПОТОК""",Признаки нелегального оператора инвестиционной...,2026-08-19,False,None
4,ООО «УК Селигер 24»,6913017372,True,True,найден в чёрном списке,"ОБЩЕСТВО С ОГРАНИЧЕННОЙ ОТВЕТСТВЕННОСТЬЮ ""УК С...",Признаки незаконного привлечения инвестиций,2026-07-30,False,None
5,ПК «Платформа Ру»,7733473794,True,True,найден в чёрном списке,"ПОТРЕБИТЕЛЬСКИЙ КООПЕРАТИВ ""ПЛАТФОРМА РУ"" (PLA...",Признаки нелегального профессионального участн...,2026-07-17,False,None
6,КПК «Сберкнижка»,6452126313,True,True,найден в чёрном списке,"КРЕДИТНЫЙ ПОТРЕБИТЕЛЬСКИЙ КООПЕРАТИВ ""СБЕРКНИЖКА""",Признаки нелегального кредитора; Признаки неза...,2026-05-04,False,None
7,СК «Образец»,0571011790,True,True,найден в чёрном списке,"АКЦИОНЕРНОЕ ОБЩЕСТВО СТРАХОВАЯ КОМПАНИЯ ""ОБРАЗЕЦ""",Признаки нелегальной деятельности на страховом...,2023-04-21,False,None
8,ООО «СтройМонтажСервис»,7707083893,True,True,не найден,None,,None,None,None
9,АО «ТранспортЛизинг»,5029066329,True,False,не найден,None,,None,None,None


---
## Блок 2. Курсовые данные

Для расчётов и отчёта нужен курс ЦБ на дату платежа каждого контрагента.

Что нужно сделать:

1. **Собрать уникальные даты платежей** реестра (без повторов) — и запрашивать курс один раз на дату, а не на каждую запись.
2. **Получить курс USD** на каждую дату через официальный источник ЦБ.
3. **Зафиксировать обе даты**: какую дату вы запросили и какую дату курс ЦБ **реально** вернул. Для части дат они не совпадут — это не ошибка, это свойство источника: курсы публикуются только по рабочим дням, и на выходной/праздник ЦБ возвращает курс последнего торгового дня. Оба значения должны попасть в результат.
4. **Сопоставить** каждому контрагенту курс его даты платежа (и фактическую дату курса).

После блока у каждой записи реестра должны появиться поля с курсом и фактической датой курса. Выведите таблицу «запрошенная дата → фактическая дата → курс USD» и отметьте, для каких дат курс сдвинулся.

In [6]:
# ===================== БЛОК 2 — ВАШ КОД =====================
from datetime import datetime
RATES_URL = "https://www.cbr.ru/scripts/XML_daily.asp"

# 1. Уникальные даты платежей реестра
unique_payment_dates = sorted(
    {cp["payment_date"] for cp in counterparties},
    key=lambda value: datetime.strptime(value, "%d.%m.%Y"),
)

# 2. Курс USD на каждую дату (официальный источник ЦБ)
# 3. Запрошенная дата + фактическая дата курса
# Получает курс USD и фактическую дату курса ЦБ в формате ДД.ММ.ГГГГ.
def fetch_usd_rate(payment_date):
    # Меняем точки на /, потому что API ЦБ ожидает именно такой формат
    api_date = payment_date.replace(".", "/")
    result = {
        "requested_date": payment_date,
        "actual_date": None,
        "usd_rate": None,
        "shifted": None,
        "error": None,
    }
    try:
        response = requests.get(
            RATES_URL,
            params={"date_req": api_date},
            timeout=REQUEST_TIMEOUT,
        )
        response.raise_for_status()
        response.encoding = "windows-1251"
        xml_text = response.text
        # Берём дату из корневого тега, а курс из блока USD
        actual_match = re.search(r'<ValCurs[^>]*Date="([\d.]+)"', xml_text)
        usd_match = re.search(
            r"<CharCode>USD</CharCode>.*?<Value>([\d,]+)</Value>",
            xml_text,
            re.S,
        )
        if actual_match is None or usd_match is None:
            raise ValueError("В XML-ответе не найдены дата или курс USD")
        actual_date = actual_match.group(1)
        usd_rate = float(usd_match.group(1).replace(",", "."))
        result.update({
            "actual_date": actual_date,
            "usd_rate": usd_rate,
            "shifted": actual_date != payment_date,
        })
        return result
    except (RequestException, ValueError, TypeError) as error:
        result["error"] = f"{type(error).__name__}: {error}"
        return result

rates_by_date = {}
for payment_date in unique_payment_dates:
    # Сохраняем ответ в словарь по дате чтобы не делать запрос еще раз
    rates_by_date[payment_date] = fetch_usd_rate(payment_date)
    time.sleep(REQUEST_DELAY)  # делаем небольшую паузу между запросами

# 4. Сопоставление курса каждому контрагенту
for counterparty in counterparties:
    rate_info = rates_by_date[counterparty["payment_date"]]
    counterparty["rate_requested_date"] = rate_info["requested_date"]
    counterparty["rate_actual_date"] = rate_info["actual_date"]
    counterparty["usd_rate"] = rate_info["usd_rate"]
    counterparty["rate_shifted"] = rate_info["shifted"]
    counterparty["rate_error"] = rate_info["error"]
print(f"Контрагентов: {len(counterparties)}")
print(f"Уникальных дат и запросов курса: {len(unique_payment_dates)}")
rates_table = pd.DataFrame([
    {
        "Запрошенная дата": info["requested_date"],
        "Фактическая дата курса": info["actual_date"],
        "Курс USD": info["usd_rate"],
        "Курс взят за предыдущую дату": info["shifted"],
        "Ошибка": info["error"],
    }
    for info in rates_by_date.values()
])
display(rates_table)


Контрагентов: 30
Уникальных дат и запросов курса: 29


,Запрошенная дата,Фактическая дата курса,Курс USD,Курс взят за предыдущую дату,Ошибка
0,14.01.2025,14.01.2025,102.7081,False,None
1,07.02.2025,07.02.2025,96.9104,False,None
2,23.02.2025,22.02.2025,88.1676,True,None
3,01.03.2025,01.03.2025,88.2568,False,None
4,10.03.2025,08.03.2025,89.1362,True,None
5,17.03.2025,15.03.2025,85.5694,True,None
6,12.04.2025,12.04.2025,84.0040,False,None
7,15.04.2025,15.04.2025,82.7671,False,None
8,26.04.2025,26.04.2025,82.6549,False,None
9,28.04.2025,26.04.2025,82.6549,True,None


---
## Блок 3. Правила риска

Присвойте каждому контрагенту **уровень риска**: `критический`, `высокий`, `средний`, `низкий`.

Систему правил вы разрабатываете **полностью сами** — это центральная часть задания.
**Что такое правило риска.** Правило — это условие на данных контрагента (и результатах Блоков 1–2), при котором сделка требует дополнительной проверки или отказа от неё. Хорошие правила опираются на юридическую логику: профиль мошенничества из чёрного списка, тип договора, масштаб обязательства, существенность контрагента для компании.

**Требования к системе:**

1. **Минимум четыре правила**, покрывающие все четыре уровня (`критический`, `высокий`, `средний`, `низкий`). Каждый уровень должен быть достижим хотя бы для одного контрагента реестра.
2. **Порядок старшинства**: если контрагенту подходят несколько правил, применяется самый высокий уровень. Порядок зафиксируйте в комментарии.
3. **Оформление каждого правила**: формулировка (комментарий перед кодом) → обоснование — почему это риск с точки зрения юриста (1–2 предложения) → реализация в коде.
4. **Осмысленность**: правила должны опираться на данные реестра и результаты Блоков 1–2. Правило, которое не зависит ни от чего и делит реестр на «нравится/не нравится», не засчитывается.

**Подсказки к направлениям** (не обязательные к использованию — это ориентиры, а не шаблон):

- сочетание типа договора с профилем контрагента (например, что означает для компании заём с организацией, у которой ЦБ выявил признаки нелегальной кредиторской деятельности?);
- масштаб обязательства (сумма) и как он влияет на приоритет проверки;
- состояние организации из чёрного списка (ликвидирована или действует) и что это меняет;
- сочетание нескольких факторов одновременно — правила-множители, повышающие уровень при совместном действии условий.

После блока у каждой записи реестра должно появиться поле с уровнем риска. Выведите распределение контрагентов по уровням и убедитесь, что все четыре уровня достижимы.

In [11]:
# ===================== БЛОК 3 — ВАШ КОД =====================
# Проверяем уровни по старшинству: критический -> высокий -> средний -> низкий

# Cистема правил риска:
# Возвращает уровень риска и понятное юристу основание
def assign_risk(counterparty):
    amount = counterparty["amount"]

    # правило 1: критический риск
    # Считаем риск критическим для действующего участника списка ЦБ или суммы от 15 млн руб
    # Так учитываем прямой регуляторный риск и очень большую концентрацию обязательств
    active_blacklist_entry = (
        counterparty["cb_found"] is True
        and counterparty["cb_liquidated"] is not True
    )
    # Сначала проверяем самый строгий уровень
    if active_blacklist_entry or amount >= 15_000_000:
        reasons = []
        if active_blacklist_entry:
            reasons.append("действующая запись в чёрном списке ЦБ")
        if amount >= 15_000_000:
            reasons.append("обязательство не менее 15 млн руб.")
        return "критический", "; ".join(reasons)

    # правило 2: высокий риск
    # Ставим высокий риск для записи ЦБ, сбоя проверки или суммы от 10 до 15 млн руб
    # Даже ликвидированная фирма оставляет репутационный риск а сбой надо проверить вручную
    if (
        counterparty["cb_found"] is True
        or counterparty["cb_status"] in {"ошибка API", "некорректный ИНН"}
        or counterparty["rate_error"] is not None
        or amount >= 10_000_000
    ):
        reasons = []
        if counterparty["cb_found"] is True:
            reasons.append("запись в чёрном списке ЦБ (в том числе ликвидированная)")
        if counterparty["cb_status"] in {"ошибка API", "некорректный ИНН"}:
            reasons.append("автоматическая проверка не завершена")
        if counterparty["rate_error"] is not None:
            reasons.append("курс ЦБ не получен автоматически")
        if amount >= 10_000_000:
            reasons.append("обязательство не менее 10 млн руб.")
        return "высокий", "; ".join(reasons)

    # правило 3: средний риск
    # Ставим средний риск для суммы от 5 млн руб либо для договоров финансового характера
    financial_contracts = {"заём", "инвестиции", "лизинг", "страхование"}
    if (
        amount >= 5_000_000
        or counterparty["contract_type"] in financial_contracts
    ):
        reasons = []
        if amount >= 5_000_000:
            reasons.append("обязательство не менее 5 млн руб.")
        if counterparty["contract_type"] in financial_contracts:
            reasons.append("финансовый характер договора")
        return "средний", "; ".join(reasons)

    # Правило 4: низкий риск
    # Оставляем низкий риск если контрагент чистый сумма меньше 5 млн. Получается что по выбранным факторам повышенного юридического риска мы не нашли
    return "низкий", "повышающие риск факторы не выявлены"

for counterparty in counterparties:
    # Вместе с уровнем сохраняем причину чтобы потом показать её в таблице
    counterparty["risk_level"], counterparty["risk_reason"] = assign_risk(counterparty)
risk_order = ["критический", "высокий", "средний", "низкий"]
risk_counts = Counter(cp["risk_level"] for cp in counterparties)
risk_amounts = {
    level: sum(cp["amount"] for cp in counterparties if cp["risk_level"] == level)
    for level in risk_order
}

# Считаем отдельно количество и сумму обязательств для итоговой сводки
risk_distribution = pd.DataFrame([
    {
        "Уровень риска": level,
        "Контрагентов": risk_counts[level],
        "Сумма обязательств, руб.": risk_amounts[level],
    }
    for level in risk_order
])
display(risk_distribution)

missing_levels = [level for level in risk_order if risk_counts[level] == 0]
# Проверяем что в результате действительно получились все четыре уровня
if missing_levels:
    print("Внимание: на текущих ответах API не представлены уровни:", ", ".join(missing_levels))
else:
    print("Проверка пройдена: в реестре представлены все четыре уровня риска.")


,Уровень риска,Контрагентов,"Сумма обязательств, руб."
0,критический,10,81300000
1,высокий,3,35100000
2,средний,8,57650000
3,низкий,9,24910000


Проверка пройдена: в реестре представлены все четыре уровня риска.


---
## Блок 4. Материалы для собрания

Подготовьте два артефакта.

**4.1. Excel-дашборд** `screening_dashboard.xlsx` (библиотека openpyxl), три листа:

- **«Сводка»** — ключевые числа: всего контрагентов; сколько в чёрном списке; распределение по уровням риска; суммарная сумма обязательств по каждому уровню риска. Числа должны сходиться с реестром.
- **«Реестр»** — все 12 контрагентов с колонками: название, ИНН, регион, тип договора, сумма, дата платежа, статус проверки ЦБ, уровень риска, курс на дату платежа. Строки с риском `критический` — красная заливка, `высокий` — жёлтая, остальные — без заливки. Шапка — жирным, ширина колонок подобрана так, чтобы названия читались.
- **«Курсы»** — таблица «запрошенная дата → фактическая дата → курс USD» по всем уникальным датам.

**4.2. Текстовое резюме** `screening_summary.txt` (5–8 строк, с числами):

- сколько контрагентов проверено;
- сколько обнаружено в чёрном списке (и сколько из них ликвидировано);
- сколько контрагентов каждого уровня риска;
- суммарная экспозиция по критическому и высокому риску (в рублях);
- сколько дат платежей пришлось на выходные/праздники (курс сдвинулся).

Резюме должно читаться как вывод для партнёров: короткие утверждения с числами, без технических деталей.

In [12]:
# ===================== БЛОК 4 — ВАШ КОД =====================
from pathlib import Path
from openpyxl import Workbook
from openpyxl.chart import BarChart, Reference
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter

DASHBOARD_FILE = Path("screening_dashboard.xlsx")
SUMMARY_FILE = Path("screening_summary.txt")

# Применяет единое оформление к табличному листу
def format_worksheet(worksheet, widths):
    header_fill = PatternFill("solid", fgColor="1F4E78")
    for cell in worksheet[1]:
        cell.font = Font(bold=True, color="FFFFFF")
        cell.fill = header_fill
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)

    for column_number, width in enumerate(widths, start=1):
        worksheet.column_dimensions[get_column_letter(column_number)].width = width

    worksheet.freeze_panes = "A2"
    worksheet.auto_filter.ref = worksheet.dimensions
total_count = len(counterparties)
# Считаем показатели из готового реестра чтобы числа в отчетах совпадали
found_count = sum(cp["cb_found"] is True for cp in counterparties)
liquidated_count = sum(
    cp["cb_found"] is True and cp["cb_liquidated"] is True
    for cp in counterparties
)
failed_checks = sum(
    cp["cb_status"] in {"ошибка API", "некорректный ИНН"}
    for cp in counterparties
)
shifted_dates_count = sum(info["shifted"] is True for info in rates_by_date.values())

# 4.1. Excel-дашборд: листы «Сводка», «Реестр», «Курсы»
workbook = Workbook()

# Лист 1 "Сводка". Оставляем тут основные показатели чтобы их можно было быстро посмотреть
summary_sheet = workbook.active
summary_sheet.title = "Сводка"
summary_sheet.append(["Показатель", "Значение"])
summary_sheet.append(["Всего контрагентов", total_count])
summary_sheet.append(["Найдено в чёрном списке ЦБ", found_count])
summary_sheet.append(["Ликвидировано из найденных", liquidated_count])
summary_sheet.append(["Сбоев проверки", failed_checks])
summary_sheet.append(["Дат со сдвигом курса", shifted_dates_count])
summary_sheet.append([])
summary_sheet.append(["Уровень риска", "Количество", "Сумма обязательств, руб."])
for level in risk_order:
    summary_sheet.append([level.capitalize(), risk_counts[level], risk_amounts[level]])
for cell in summary_sheet[1]:
    cell.font = Font(bold=True, color="FFFFFF")
    cell.fill = PatternFill("solid", fgColor="1F4E78")
for cell in summary_sheet[8]:
    cell.font = Font(bold=True, color="FFFFFF")
    cell.fill = PatternFill("solid", fgColor="1F4E78")
summary_sheet.column_dimensions["A"].width = 38
summary_sheet.column_dimensions["B"].width = 18
summary_sheet.column_dimensions["C"].width = 28
for row in range(9, 13):
    summary_sheet.cell(row=row, column=3).number_format = '#,##0 "₽"'
# Добавим диаграмму по уровням риска
chart = BarChart()
chart.title = "Распределение контрагентов по риску"
chart.y_axis.title = "Количество"
chart.x_axis.title = "Уровень риска"
chart.add_data(Reference(summary_sheet, min_col=2, min_row=8, max_row=12), titles_from_data=True)
chart.set_categories(Reference(summary_sheet, min_col=1, min_row=9, max_row=12))
chart.height = 7
chart.width = 12
summary_sheet.add_chart(chart, "E2")

# Лист 2 "Реестр". Сохраняем обязательные колонки и дополнительно основание уровня риска
registry_sheet = workbook.create_sheet("Реестр")
registry_headers = [
    "Название", "ИНН", "Регион", "Тип договора", "Сумма, руб.",
    "Дата платежа", "Статус проверки ЦБ", "Уровень риска",
    "Курс USD", "Фактическая дата курса", "Основание риска",
    "Название в реестре ЦБ", "Признаки ЦБ", "Дата внесения", "Ликвидирована",
]
registry_sheet.append(registry_headers)
for counterparty in counterparties:
    registry_sheet.append([
        counterparty["name"],
        counterparty["inn_clean"],
        counterparty["region"],
        counterparty["contract_type"],
        counterparty["amount"],
        counterparty["payment_date"],
        counterparty["cb_status"],
        counterparty["risk_level"],
        counterparty["usd_rate"],
        counterparty["rate_actual_date"],
        counterparty["risk_reason"],
        counterparty["cb_name"],
        "; ".join(counterparty["cb_signs"]),
        counterparty["cb_listed_date"],
        counterparty["cb_liquidated"],
    ])

critical_fill = PatternFill("solid", fgColor="F4CCCC")
high_fill = PatternFill("solid", fgColor="FFF2CC")
for row_number, counterparty in enumerate(counterparties, start=2):
    fill = None
    if counterparty["risk_level"] == "критический":
        fill = critical_fill
    elif counterparty["risk_level"] == "высокий":
        fill = high_fill
    if fill:
        for cell in registry_sheet[row_number]:
            cell.fill = fill
    registry_sheet.cell(row=row_number, column=5).number_format = '#,##0 "₽"'
    registry_sheet.cell(row=row_number, column=9).number_format = "0.0000"
    for cell in registry_sheet[row_number]:
        cell.alignment = Alignment(vertical="top", wrap_text=True)
format_worksheet(
    registry_sheet,
    [34, 14, 25, 16, 16, 15, 24, 16, 12, 20, 48, 38, 55, 15, 15],
)

# Лист 3 "Курсы". Показываем какой курс использовали для каждой уникальной даты
rates_sheet = workbook.create_sheet("Курсы")
rates_sheet.append([
    "Запрошенная дата", "Фактическая дата", "Курс USD", "Сдвиг даты", "Ошибка"
])
for payment_date in unique_payment_dates:
    info = rates_by_date[payment_date]
    rates_sheet.append([
        info["requested_date"],
        info["actual_date"],
        info["usd_rate"],
        "да" if info["shifted"] is True else ("нет" if info["shifted"] is False else "неизвестно"),
        info["error"],
    ])
    rates_sheet.cell(row=rates_sheet.max_row, column=3).number_format = "0.0000"
format_worksheet(rates_sheet, [20, 20, 14, 14, 55])
workbook.save(DASHBOARD_FILE)

# 4.2. Текстовое резюме screening_summary.txt
# Делаем резюме из 7 коротких строк для собрания, используя те же показатели что и для Excel
critical_exposure = risk_amounts["критический"]
high_exposure = risk_amounts["высокий"]
summary_lines = [
    f"Проведён скрининг {total_count} контрагентов; незавершённых автоматических проверок: {failed_checks}.",
    f"В чёрном списке Банка России обнаружено {found_count} контрагентов, из них ликвидировано: {liquidated_count}.",
    "Распределение риска: " + ", ".join(f"{level} — {risk_counts[level]}" for level in risk_order) + ".",
    f"Экспозиция по критическому риску: {critical_exposure:,.0f} руб.",
    f"Экспозиция по высокому риску: {high_exposure:,.0f} руб.",
    f"Совокупная экспозиция критического и высокого риска: {critical_exposure + high_exposure:,.0f} руб.",
    f"Курс ЦБ был сдвинут на предыдущую опубликованную дату для {shifted_dates_count} из {len(unique_payment_dates)} уникальных дат платежей.",
]
SUMMARY_FILE.write_text("\n".join(summary_lines), encoding="utf-8-sig")

print(f"Создан Excel-дашборд: {DASHBOARD_FILE.resolve()}")
print(f"Создано текстовое резюме: {SUMMARY_FILE.resolve()}")
print("\n" + "\n".join(summary_lines))


Создан Excel-дашборд: /content/screening_dashboard.xlsx
Создано текстовое резюме: /content/screening_summary.txt

Проведён скрининг 30 контрагентов; незавершённых автоматических проверок: 0.
В чёрном списке Банка России обнаружено 8 контрагентов, из них ликвидировано: 0.
Распределение риска: критический — 10, высокий — 3, средний — 8, низкий — 9.
Экспозиция по критическому риску: 81,300,000 руб.
Экспозиция по высокому риску: 35,100,000 руб.
Совокупная экспозиция критического и высокого риска: 116,400,000 руб.
Курс ЦБ был сдвинут на предыдущую опубликованную дату для 13 из 29 уникальных дат платежей.


---
## Критерии оценивания

Максимум — **8 баллов**. Оценивается каждый блок отдельно; общие требования к коду влияют на итоговую оценку.

### Блок 1. Скрининг по чёрному списку — до 2 баллов

| Критерий | Баллы |
|----------|-------|
| Все ИНН очищены (`inn_clean`), «мусор» обработан корректно (включая ведущий ноль и пробелы) | 1 |
| Реализована цепочка Search → DetailInfo → **точная сверка ИНН** (не только совпадение по фразе); у всех 30 записей есть результат проверки — для найденных: название, признаки, дата внесения, статус ликвидации; сетевые сбои не прерывают обработку, статистика выведена; у всех запросов `timeout` и `raise_for_status()`, между запросами паузы | 1 |

### Блок 2. Курсовые данные — до 2 баллов

| Критерий | Баллы |
|----------|-------|
| Даты собраны как множество уникальных: запросов к ЦБ меньше, чем записей реестра; курс получен с официального источника, кодировка windows-1251 обработана, числа корректны (запятая → точка) | 1 |
| Для каждой даты зафиксированы запрошенная **и** фактическая даты, расхождения отражены в таблице; курс и фактическая дата сопоставлены каждому контрагенту | 1 |

### Блок 3. Правила риска — до 2 баллов

| Критерий | Баллы |
|----------|-------|
| Минимум 4 правила, покрывающие все четыре уровня; каждый уровень достижим на реестре; порядок старшинства определён и зафиксирован | 1 |
| Каждое правило оформлено (формулировка → обоснование → код); обоснования содержательны — юридическая логика, а не пересказ условия; правила опираются на данные реестра и результаты Блоков 1–2 | 1 |

### Блок 4. Материалы для собрания — до 2 баллов

| Критерий | Баллы |
|----------|-------|
| Excel-дашборд `screening_dashboard.xlsx`: лист «Сводка» с числами, сходящимися с реестром; лист «Реестр» со всеми контрагентами, колонками и заливкой по уровню риска; лист «Курсы» со всеми датами и обеими датами | 1 |
| Текстовое резюме `screening_summary.txt`: 5–8 строк, все требуемые числа, читается как вывод для партнёров | 1 |


За технические и стилистические ошибки можно получить "штрафы" по баллам, которые повлияют на итоговую оценку.

**Итоговая оценка**: сумма баллов четырёх блоков минус штрафы, максимум 8.

